In [1]:
import sys
import numpy as np
import matplotlib.pyplot as plt

sys.path.append("/mnt/lareaulab/reliscu/code")

from parse_gtf import *

In [7]:
ctype_abund_df = pd.read_csv("data/hahn_2023_cortex_STAR_counts_TMMF_All_130_outliers_removed_44764genes_cleaned_mergeParam0.93_minSignum0.88_Modules_top_corr_enriched_w_Claude_marker_genes_PC1_ctype_abundance.csv", index_col=0)

In [3]:
# Parse GTF attribute column
gtf_file = "/mnt/lareaulab/reliscu/data/GENCODE/GRCm39/gencode.vM35.annotation.gtf"
gtf = gtf_parse(gtf_file)
gtf_subset = gtf.loc[gtf['feature'].isin(["gene"])]
attrs = gtf_subset["attribute"].apply(extract_attributes)
attrs_df = attrs.apply(pd.Series)
gtf_parsed = pd.concat([gtf_subset.drop(columns=["attribute"]), attrs_df], axis=1)
gtf_parsed['gene_id'] = gtf_parsed['gene_id'].str.split(".").str[0]

# Gene expr

In [9]:
bulk_expr = pd.read_csv("data/cleaned/ModSeed/hahn_2023_cortex_STAR_counts_TMMF_All_130_outliers_removed_44764genes_cleaned.csv")
bulk_expr.columns.values[0] = "Gene"

In [10]:
mean_expr = pd.DataFrame({
    'Index': range(len(bulk_expr)),
    'Gene': bulk_expr.iloc[:, 0],
    'Expr': bulk_expr.iloc[:, 1:].mean(axis=1)
})

keep = mean_expr.loc[mean_expr.groupby('Gene')['Expr'].idxmax(), 'Index']
bulk_expr_filtered = bulk_expr.iloc[keep.values]

In [11]:
bulk_expr_filtered = bulk_expr_filtered.set_index("Gene")
common = bulk_expr_filtered.columns.intersection(ctype_abund_df.index)
bulk_expr_filtered = bulk_expr_filtered[common]
ctype_abund_df = ctype_abund_df.loc[common]

In [12]:
# Correlate each cell type with all PSI events across samples

# expr_numeric = bulk_expr_filtered.iloc[:, 1:].apply(pd.to_numeric, errors='coerce')

expr_corr_results = {}
for ct in ctype_abund_df.columns:
    expr_corr_results[ct] = bulk_expr_filtered.T.corrwith(ctype_abund_df[ct])

/mnt/lareaulab/reliscu/anaconda3/envs/anndata/lib/python3.12/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/mnt/lareaulab/reliscu/anaconda3/envs/anndata/lib/python3.12/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/mnt/lareaulab/reliscu/anaconda3/envs/anndata/lib/python3.12/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/mnt/lareaulab/reliscu/anaconda3/envs/anndata/lib/python3.12/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/mnt/lareaulab/reliscu/anaconda3/envs/anndata/lib/python3.12/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/mnt/lareaulab/reliscu/anaconda3/envs/anndata/lib/python3.12/site-packages/numpy/lib/

In [13]:
expr_corr_df = pd.DataFrame(expr_corr_results)
expr_corr_df.head()

,All GABAergic,All Glutamatergic,All Neuronal,Astro,CGE Class,Chandelier,Endo,L2/3 IT,L4 IT,L5 ET,...,OPC,Oligo,Pax6,Peri,Pvalb,Sncg,Sst,Sst Chodl,VLMC,Vip
Gene,,,,,,,,,,,,,,,,,,,,,
0610005C13Rik,0.648076,0.525020,0.526419,0.658197,0.634063,0.646719,0.599301,0.495451,0.425887,0.475671,...,0.612428,0.314180,0.659744,0.646719,0.646719,0.612911,0.641046,0.612911,0.630474,0.452894
0610006L08Rik,0.210448,0.141898,0.127303,0.222243,0.223589,0.210617,0.182926,0.224374,0.289187,0.150099,...,0.136878,0.039092,0.225663,0.210617,0.210617,0.228154,0.227326,0.228154,0.309597,0.185302
0610009E02Rik,0.904715,0.761939,0.768543,0.913521,0.884050,0.900938,0.838792,0.694968,0.575596,0.698825,...,0.848039,0.553434,0.913919,0.900938,0.900938,0.845958,0.890905,0.845958,0.772615,0.624183
0610009L18Rik,0.890585,0.649157,0.670198,0.925060,0.959815,0.878517,0.754864,0.543074,0.445396,0.858353,...,0.916645,0.477052,0.919181,0.878517,0.878517,0.950480,0.960468,0.950480,0.834319,0.445222
0610010K14Rik,0.800898,0.711606,0.710362,0.796186,0.727149,0.802865,0.778175,0.681306,0.597851,0.566160,...,0.720908,0.513917,0.800923,0.802865,0.802865,0.682315,0.740205,0.682315,0.719629,0.637115


In [14]:
expr_corr_df.to_csv(f"data/corrs/hahn_2023_cortex_STAR_counts_TMMF_All_130_outliers_removed_44764genes_cleaned_mergeParam0.93_minSignum0.88_Modules_top_corr_enriched_w_Claude_marker_genes_PC1_ctype_abundance_gene_expr_corr.csv")